# RCS-55 — Baseline règles/ML reproductible

## Fiche Jira (reprise fidèle)
- **Tâche** : RCS-55 — « Dataset indépendant, protocole, métriques, seuils et splits. »
- **Épic** : RCS-8 ; **phase** : S3 ; **priorité** : High ; **statut** : À faire.
- **Critère d'acceptation** : « Notebook baseline reproductible ».
- **Compétence** : Machine Learning.
- **Rôles** : Elias (responsable), Ayoub (binôme), Omar (relais), Gémima (relectrice).
- **Source** : bibliothèque RoboCare Sentinel V3. Aucune date ni estimation n'est arrêtée : ne rien inventer.

Ce notebook rejoue le protocole et affiche les mesures calculées à l'exécution. Il ne contient aucune valeur de métrique figée.

## Périmètre et frontière des preuves
- Le jeu cible est un **dataset indépendant** (D-002 dans `data/manifest.md`) à sélectionner, avec origine, droits et séparation train/val/test documentés. Tant qu'il n'est pas gelé, les seules mesures reproductibles sont celles d'un **smoke test synthétique**, étiqueté **non médical**.
- Le mode `--synthetic` valide le pipeline ; il ne prouve aucune performance clinique.
- `ND` = non disponible, et non égal à zéro.

## Protocole
1. Charger le dataset indépendant (ou générer le jeu synthétique de smoke test), sans mélanger les sources.
2. Split **train/test stratifié** reproductible : `seed=42`, `test_size=0.2` (valeurs par défaut du module, modifiables explicitement).
3. **Anti-fuite** : le scaler de la baseline règles est ajusté sur le TRAIN uniquement ; sa moyenne et son écart-type train (ddof=0) sont appliqués au TEST. Aucune statistique n'est recalculée sur le TEST ni sur TRAIN+TEST. Les labels du TEST servent uniquement à l'évaluation.
4. IsolationForest est entraîné sur le TRAIN sans labels, puis évalué sur le TEST.
5. Métriques : **precision**, **recall**, **F1** et **matrice de confusion** (labels `normal` / `anomaly`).
6. Seuils : le z-score est calibré sur le TRAIN ; `z_threshold` et la `contamination` IsolationForest restent des paramètres de configuration, **non gelés** ; le **seuil de décision final est `ND`** avant campagne.

In [ ]:
from pathlib import Path
import json
import sys

RACINE = Path.cwd().resolve()
if not (RACINE / "experiments" / "baseline_rules_ml.py").exists():
    RACINE = RACINE.parent
sys.path.insert(0, str(RACINE))

from experiments.baseline_rules_ml import (
    RunConfig, load_dataset, stratified_split, fit_rule_calibration,
    rule_based_predict, evaluate, run,
)

print(f"Racine du dépôt : {RACINE}")

## Configuration de la rejouabilité
La configuration ci-dessous reprend les valeurs par défaut du module. Aucune métrique n'est écrite ici : tous les affichages sont calculés aux cellules suivantes.

In [ ]:
cfg = RunConfig(synthetic=True)  # smoke test : remplacer par RunConfig(data_path="...") pour un dataset indépendant
print(json.dumps(cfg.__dict__, indent=2, ensure_ascii=False))

## Chargement, split et calibration sur le train uniquement

In [ ]:
df = load_dataset(cfg)
X = df.drop(columns=["label"])
y = df["label"].astype(int).to_numpy()
X_train, X_test, y_train, y_test = stratified_split(X, y, cfg)
calibration = fit_rule_calibration(X_train)

print(json.dumps({
    "lignes": int(len(df)),
    "lignes_train": int(len(X_train)),
    "lignes_test": int(len(X_test)),
    "distribution_labels": {str(k): int(v) for k, v in df["label"].value_counts().items()},
    "calibration_fitted_on": calibration.as_dict()["fitted_on"],
    "moyennes_train": calibration.means,
    "ecarts_types_train": calibration.stds,
}, indent=2, ensure_ascii=False))

## Baseline règles — z-score calibré sur le train
La décision sur le TEST utilise les statistiques TRAIN ci-dessus, sans recalcul.

In [ ]:
y_pred_rules = rule_based_predict(X_test, calibration, cfg.z_threshold)
resultats_rules = evaluate(y_test, y_pred_rules)
print(json.dumps(resultats_rules, indent=2, ensure_ascii=False))

## Baseline ML — IsolationForest
Le pipeline complet (`run`) applique le même split et la même règle de calibration ; il produit la structure d'artefact versée dans `experiments/results/`.

In [ ]:
resultats = run(cfg)
print(json.dumps(resultats["isolation_forest_baseline"], indent=2, ensure_ascii=False))

## Comparaison et seuils
Les valeurs de seuil affichées sont celles de la configuration du smoke test ; aucune n'est un seuil de détection gelé.

In [ ]:
cles = ("precision", "recall", "f1", "confusion_matrix")
print(json.dumps({
    "regles": {cle: resultats_rules[cle] for cle in cles},
    "isolation_forest": {cle: resultats["isolation_forest_baseline"][cle] for cle in cles},
    "protocole": resultats["protocol"],
    "seuils": resultats["thresholds"],
    "statut": resultats["status"],
}, indent=2, ensure_ascii=False))

## Limites
- Smoke test synthétique : jeu **non médical**, classes artificiellement séparables, aucune validation clinique.
- Aucun seuil de détection gelé : `z_threshold` et `contamination` sont des paramètres de configuration ; le seuil de décision final reste `ND` avant campagne.
- Résultats réels uniquement : les chiffres affichés sont calculés à l'exécution sur les données chargées/générées ; aucune valeur de métrique n'est figée dans ce notebook.
- Le dataset indépendant, les droits, les splits et les seuils doivent être formalisés avant toute campagne.